In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# بناء الدوائر

<details>
<summary><b>إصدارات الحزم</b></summary>

تم تطوير الكود في هذه الصفحة باستخدام المتطلبات التالية.
نوصي باستخدام هذه الإصدارات أو أحدث منها.

```
qiskit[all]~=2.3.0
```
</details>

تتناول هذه الصفحة بشكل أعمق فئة [`QuantumCircuit`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit) في Qiskit SDK، وتستعرض بعض الأساليب الأكثر تقدمًا التي يمكنك استخدامها لإنشاء الدوائر الكمومية.

## ما هي الدائرة الكمومية؟
الدائرة الكمومية البسيطة هي مجموعة من Qubits وقائمة من التعليمات التي تعمل عليها. لتوضيح ذلك، ينشئ الكود التالي دائرة جديدة بـ Qubitين، ثم يعرض السمة [`qubits`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit#qubits) الخاصة بالدائرة، وهي قائمة من [`Qubits`](https://docs.quantum.ibm.com/api/qiskit/circuit#qiskit.circuit.Qubit) مرتبة من البت الأقل أهمية $q_0$ إلى البت الأكثر أهمية $q_n$.

In [1]:
from qiskit import QuantumCircuit

qc = QuantumCircuit(2)
qc.qubits

[<Qubit register=(2, "q"), index=0>, <Qubit register=(2, "q"), index=1>]

Multiple `QuantumRegister` and `ClassicalRegister` objects can be combined to create a circuit. Every [`QuantumRegister`](/docs/api/qiskit/circuit#qiskit.circuit.QuantumRegister) and [`ClassicalRegister`](/docs/api/qiskit/circuit#qiskit.circuit.ClassicalRegister) can also be named.

In [2]:
from qiskit.circuit import QuantumRegister, ClassicalRegister

qr1 = QuantumRegister(2, "qreg1")  # Create a QuantumRegister with 2 qubits
qr2 = QuantumRegister(1, "qreg2")  # Create a QuantumRegister with 1 qubit
cr1 = ClassicalRegister(3, "creg1")  # Create a ClassicalRegister with 3 cbits

combined_circ = QuantumCircuit(
    qr1, qr2, cr1
)  # Create a quantum circuit with 2 QuantumRegisters and 1 ClassicalRegister
combined_circ.qubits

[<Qubit register=(2, "qreg1"), index=0>,
 <Qubit register=(2, "qreg1"), index=1>,
 <Qubit register=(1, "qreg2"), index=0>]

يمكن دمج كائنات `QuantumRegister` و`ClassicalRegister` المتعددة لإنشاء دائرة. ويمكن تسمية كل [`QuantumRegister`](https://docs.quantum.ibm.com/api/qiskit/circuit#qiskit.circuit.QuantumRegister) و[`ClassicalRegister`](https://docs.quantum.ibm.com/api/qiskit/circuit#qiskit.circuit.ClassicalRegister) أيضًا.

In [3]:
desired_qubit = qr2[0]  # Qubit 0 of register 'qreg2'

print("Index:", combined_circ.find_bit(desired_qubit).index)
print("Register:", combined_circ.find_bit(desired_qubit).registers)

Index: 2
Register: [(QuantumRegister(1, 'qreg2'), 0)]


Adding an instruction to the circuit appends the instruction to the circuit's [`data`](/docs/api/qiskit/qiskit.circuit.QuantumCircuit#data) attribute. The following cell output shows `data` is a list of [`CircuitInstruction`](/docs/api/qiskit/qiskit.circuit.CircuitInstruction) objects, each of which has an `operation` attribute, and a `qubits` attribute.

In [4]:
qc.x(0)  # Add X-gate to qubit 0
qc.data

[CircuitInstruction(operation=Instruction(name='x', num_qubits=1, num_clbits=0, params=[]), qubits=(<Qubit register=(2, "q"), index=0>,), clbits=())]

يمكنك معرفة فهرس Qubit وسجله باستخدام طريقة [`find_bit`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit#qiskit.circuit.QuantumCircuit.find_bit) الخاصة بالدائرة وسماتها.

In [5]:
qc.draw("mpl")

<Image src="../docs/images/guides/construct-circuits/extracted-outputs/43a57258-3e33-4071-8a48-2bf127c8a5be-0.svg" alt="Output of the previous code cell" />

Circuit instruction objects can contain "definition" circuits that describe the instruction in terms of more fundamental instructions. For example, the [X-gate](/docs/api/qiskit/qiskit.circuit.library.XGate) is defined as a specific case of the [U3-gate](/docs/api/qiskit/qiskit.circuit.library.U3Gate), a more general single-qubit gate.

In [6]:
# Draw definition circuit of 0th instruction in `qc`
qc.data[0].operation.definition.draw("mpl")

<Image src="../docs/images/guides/construct-circuits/extracted-outputs/653e2427-e301-4d2f-84de-1959185ace8e-0.svg" alt="Output of the previous code cell" />

إضافة تعليمة إلى الدائرة يُلحق التعليمة بسمة [`data`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit#data) الخاصة بالدائرة. يُظهر ناتج الكود التالي أن `data` هي قائمة من كائنات [`CircuitInstruction`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.CircuitInstruction)، ولكل منها سمة `operation` وسمة `qubits`.

In [7]:
from qiskit.circuit.library import HGate

qc = QuantumCircuit(1)
qc.append(
    HGate(),  # New HGate instruction
    [0],  # Apply to qubit 0
)
qc.draw("mpl")

<Image src="../docs/images/guides/construct-circuits/extracted-outputs/66813cae-9841-47ea-96b7-8fd7b82e9759-0.svg" alt="Output of the previous code cell" />

To combine two circuits, use the [`compose`](/docs/api/qiskit/qiskit.circuit.QuantumCircuit#compose) method. This accepts another [`QuantumCircuit`](/docs/api/qiskit/qiskit.circuit.QuantumCircuit) and an optional list of qubit mappings.

<Admonition type="note">
    The [`compose`](/docs/api/qiskit/qiskit.circuit.QuantumCircuit#compose) method returns a new circuit and does **not** mutate either circuit it acts on. To mutate the circuit on which you're calling the [`compose`](/docs/api/qiskit/qiskit.circuit.QuantumCircuit#compose) method, use the argument `inplace=True`.
</Admonition>

In [8]:
qc_a = QuantumCircuit(4)
qc_a.x(0)

qc_b = QuantumCircuit(2, name="qc_b")
qc_b.y(0)
qc_b.z(1)

# compose qubits (0, 1) of qc_a to qubits (1, 3) of qc_b respectively
combined = qc_a.compose(qc_b, qubits=[1, 3])
combined.draw("mpl")

<Image src="../docs/images/guides/construct-circuits/extracted-outputs/29152dfa-2275-4bc4-aadb-82185b9e0e86-0.svg" alt="Output of the previous code cell" />

أسهل طريقة لعرض هذه المعلومات هي استخدام طريقة [`draw`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit#draw)، التي تُعيد تمثيلًا مرئيًا للدائرة. راجع [تصور الدوائر](./visualize-circuits) للاطلاع على طرق مختلفة لعرض الدوائر الكمومية.

In [9]:
inst = qc_b.to_instruction()
qc_a.append(inst, [1, 3])
qc_a.draw("mpl")

<Image src="../docs/images/guides/construct-circuits/extracted-outputs/81b682dd-45cb-4492-809e-d9e8ebbf5600-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/construct-circuits/extracted-outputs/43a57258-3e33-4071-8a48-2bf127c8a5be-0.svg)

يمكن أن تحتوي كائنات تعليمات الدائرة على دوائر "تعريف" تصف التعليمة بمصطلحات تعليمات أكثر أساسية. على سبيل المثال، يُعرَّف [بوابة X](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.XGate) باعتبارها حالة خاصة من [بوابة U3](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.U3Gate)، وهي بوابة أكثر عمومية لـ Qubit واحد.

In [10]:
gate = qc_b.to_gate().control()
qc_a.append(gate, [0, 1, 3])
qc_a.draw("mpl")

<Image src="../docs/images/guides/construct-circuits/extracted-outputs/ed362e64-d6a4-4dfd-a5cf-5e6bdc7a81b5-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/construct-circuits/extracted-outputs/653e2427-e301-4d2f-84de-1959185ace8e-0.svg)

التعليمات والدوائر متشابهة في أنها تصف كلتاهما عمليات على البتات و Qubits، لكنها تخدم أغراضًا مختلفة:

- تُعامَل التعليمات على أنها ثابتة، وعادةً ما تُعيد طرقها تعليمات جديدة (دون تغيير الكائن الأصلي).
- صُمِّمت الدوائر للبناء عبر أسطر كثيرة من الكود، وكثيرًا ما تُعدِّل طرق [`QuantumCircuit`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit) الكائن الموجود.

### ما هو عمق الدائرة؟
[عمق](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit#qiskit.circuit.QuantumCircuit.depth) الدائرة الكمومية هو مقياس لعدد "طبقات" البوابات الكمومية المنفَّذة بالتوازي اللازمة لإتمام الحساب الذي تُعرِّفه الدائرة. لأن البوابات الكمومية تستغرق وقتًا في التنفيذ، يتوافق عمق الدائرة تقريبًا مع المدة الزمنية التي يحتاجها الحاسوب الكمومي لتنفيذها. لذا يُعدّ عمق الدائرة من أبرز المعايير التي تُحدِّد ما إذا كان بالإمكان تشغيل الدائرة على جهاز معين.

تُوضِّح بقية هذه الصفحة كيفية التعامل مع الدوائر الكمومية.

## بناء الدوائر
تُضيف طرق مثل [`QuantumCircuit.h`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit#h) و[`QuantumCircuit.cx`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit#cx) تعليمات محددة إلى الدوائر. لإضافة تعليمات إلى دائرة بشكل أعم، استخدم طريقة [`append`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit#append). تأخذ هذه الطريقة تعليمةً وقائمةً من Qubits لتطبيق التعليمة عليها. راجع [توثيق واجهة برمجة مكتبة الدوائر](https://docs.quantum.ibm.com/api/qiskit/circuit_library) للاطلاع على قائمة التعليمات المدعومة.

In [11]:
qc_a.decompose().draw("mpl")

<Image src="../docs/images/guides/construct-circuits/extracted-outputs/3c0633db-929b-4428-a888-7a3d493bd6dd-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/construct-circuits/extracted-outputs/66813cae-9841-47ea-96b7-8fd7b82e9759-0.svg)

لدمج دائرتين، استخدم طريقة [`compose`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit#compose). تقبل هذه الطريقة كائن [`QuantumCircuit`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit) آخر وقائمةً اختيارية من تعيينات Qubits.

> **Note:** تُعيد طريقة [`compose`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit#compose) دائرةً جديدة ولا تُعدِّل **أيًّا** من الدائرتين اللتين تعمل عليهما. لتعديل الدائرة التي تستدعي عليها طريقة [`compose`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit#compose)، استخدم الوسيط `inplace=True`.

In [12]:
qc1 = QuantumCircuit(2, 2)
qc1.measure(0, 1)
qc1.draw("mpl", cregbundle=False)

<Image src="../docs/images/guides/construct-circuits/extracted-outputs/0cdb2273-0.svg" alt="Output of the previous code cell" />

In [13]:
qc2 = QuantumCircuit(2)
qc2.measure_all()
qc2.draw("mpl", cregbundle=False)

<Image src="../docs/images/guides/construct-circuits/extracted-outputs/6f33698c-0.svg" alt="Output of the previous code cell" />

In [14]:
qc3 = QuantumCircuit(2)
qc3.x(1)
qc3.measure_active()
qc3.draw("mpl", cregbundle=False)

<Image src="../docs/images/guides/construct-circuits/extracted-outputs/ca3f225f-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/construct-circuits/extracted-outputs/ed362e64-d6a4-4dfd-a5cf-5e6bdc7a81b5-0.svg)

لرؤية ما يجري، يمكنك استخدام طريقة [`decompose`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit#decompose) لتوسيع كل تعليمة إلى تعريفها.

> **Note:** تُعيد طريقة [`decompose`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit#decompose) دائرةً جديدة ولا تُعدِّل الدائرة التي تعمل عليها.

In [15]:
from qiskit.transpiler import generate_preset_pass_manager
from qiskit.circuit import Parameter

angle = Parameter("angle")  # undefined number

# Create and optimize circuit once
qc = QuantumCircuit(1)
qc.rx(angle, 0)
qc = generate_preset_pass_manager(
    optimization_level=3, basis_gates=["u", "cx"]
).run(qc)

qc.draw("mpl")

<Image src="../docs/images/guides/construct-circuits/extracted-outputs/a580552c-d585-4047-99f0-32aafd06e4f3-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/construct-circuits/extracted-outputs/3c0633db-929b-4428-a888-7a3d493bd6dd-0.svg)

<span id="measure-qubits"></span>
## قياس Qubits
تُستخدم القياسات لأخذ عيِّنات من حالات Qubits الفردية ونقل النتائج إلى سجل كلاسيكي. لاحظ أنه إذا كنت تُرسل الدوائر إلى بدائيّة [Sampler](./primitives#sampler)، فإن القياسات مطلوبة. أما الدوائر المرسلة إلى بدائيّة [Estimator](./primitives#estimator) فيجب ألا تحتوي على قياسات.

يمكن قياس Qubits باستخدام ثلاث طرق: [`measure`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit#qiskit.circuit.QuantumCircuit.measure) و[`measure_all`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit#measure_all) و[`measure_active`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit#measure_active). لمعرفة كيفية تصور نتائج القياس، راجع صفحة [تصور النتائج](./visualize-results).

1. `QuantumCircuit.measure` : يقيس كل Qubit في المعامل الأول على البت الكلاسيكي المحدد في المعامل الثاني. تتيح هذه الطريقة تحكمًا كاملًا في مكان تخزين نتيجة القياس.

2. `QuantumCircuit.measure_all` : لا تأخذ أي معامل ويمكن استخدامها للدوائر الكمومية التي لا تحتوي على بتات كلاسيكية مُعرَّفة مسبقًا. تُنشئ أسلاكًا كلاسيكية وتخزن نتائج القياس بالترتيب. على سبيل المثال، يُخزَّن قياس Qubit $q_i$ في cbit$meas_i$). كما تُضيف حاجزًا قبل القياس.

3. `QuantumCircuit.measure_active` : مشابهة لـ `measure_all`، لكنها تقيس فقط Qubits التي تحتوي على عمليات.

In [16]:
circuits = []
for value in range(100):
    circuits.append(qc.assign_parameters({angle: value}))

circuits[0].draw("mpl")

<Image src="../docs/images/guides/construct-circuits/extracted-outputs/85af6231-921a-4130-99d3-f6998f761df8-0.svg" alt="Output of the previous code cell" />

You can find a list of a circuit's undefined parameters in its `parameters` attribute.

In [17]:
qc.parameters

ParameterView([Parameter(angle)])

### Change a parameter's name

By default, parameter names for a parameterized circuit are prefixed by `x`- for example, `x[0]`. You can change the names after they are defined, as shown in the following example.

In [18]:
from qiskit.circuit.library import z_feature_map
from qiskit.circuit import ParameterVector

# Define a parameterized circuit with default names
# For example, x[0]
circuit = z_feature_map(2)

# Set new parameter names
# They will now be prefixed by `hi` instead
# For example, hi[0]
training_params = ParameterVector("hi", 2)

# Assign parameter names to the quantum circuit
circuit = circuit.assign_parameters(parameters=training_params)

![Output of the previous code cell](../docs/images/guides/construct-circuits/extracted-outputs/ca3f225f-0.svg)

## الدوائر ذات المعاملات

تتضمن كثير من الخوارزميات الكمومية قصيرة المدى تنفيذ تغييرات متعددة على دائرة كمومية. بما أن بناء الدوائر الكبيرة وتحسينها قد يكون مُكلفًا حسابيًا، يدعم Qiskit الدوائر **ذات المعاملات**. هذه الدوائر تحتوي على معاملات غير مُعرَّفة، ولا يلزم تحديد قيمها إلا قُبيل تنفيذ الدائرة. يتيح لك ذلك نقل عملية بناء الدائرة وتحسينها خارج الحلقة الرئيسية للبرنامج. يُنشئ الكود التالي دائرة ذات معاملات ويعرضها.